# 1. Imports

In [ ]:
import matplotlib.pylab as plt
import pandas as pd
import seaborn as sns

plt.style.use("dark_background")
# pd.set_option('max_columns', 100)
plt.rcParams["figure.figsize"] = (14, 9)

In [ ]:
df = pd.read_csv("./downloads/fraud_enriched.csv")

# 2. Data understanding

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.describe().T

# 3. Data preparation

In [ ]:
df.head()

In [ ]:
df["event_ts"] = pd.to_datetime(df["event_ts"])

In [ ]:
df.dtypes

In [ ]:
df.columns = map(str.lower, df.columns)
df = df.rename(columns={"class": "is_fraud"})

In [ ]:
type_mapping = {
    "merchant_category_code": "category",
    "merchant_country": "category",
    "currency": "category",
    "device_type": "category",
    "entry_mode": "category",
    "channel": "category",
    "is_international": "category",
    "is_fraud": "category",
}
df = df.astype(type_mapping)

In [ ]:
df.describe()

In [ ]:
df.describe(include=["category"])

In [ ]:
df.isna().sum()

In [ ]:
display(df.loc[df.duplicated()])

columns_subsets = [
    ["transaction_id"],
    df.columns.to_list()[1:],  # All but transation_id
    df.columns.to_list()[2:30],  # Only starting v columns
    df.columns.to_list()[30:],  # All but transaction_id, time and starting v columns
]
for s in columns_subsets:
    if df.loc[df.duplicated(subset=s)].empty is True:
        print("Subset has no duplicated values")
    else:
        print(f"Check {s} subset")

In [ ]:
df.dtypes
display(df.head(5))

# 4. Feature understanding

## Univariate analysis

### Is_fraud

In [ ]:
fig, ax = plt.subplots(layout="constrained")

sns.countplot(data=df, x="is_fraud", stat="percent", ax=ax)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Not fraud", "Fraud"])

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

ax.set_ylabel("Percentage")
ax.set_title("Distribution of Fraudulent and Non-Fraudulent Transactions")
plt.show()

### time

In [ ]:
col_to_plot = "time"

fig = sns.histplot(
    df,
    x=col_to_plot,
    stat="density",
    hue="is_fraud",
    kde=True,
    bins=15,
    common_norm=False,
    legend=True,
)

fig.set_title(f"Distribution of transactions {col_to_plot}")
fig.set_xlabel(f"{col_to_plot.capitalize()}")
fig.set_ylabel("Density")
plt.show()

### V columns

In [ ]:
# Choose a range or a list of numbers between 1 and 28
# in order to not display all plots.

cols = range(1, 4)

for col in cols:
    fig = sns.histplot(
        df,
        x=f"v{col}",
        stat="density",
        hue="is_fraud",
        kde=True,
        # bins=15,
        common_norm=False,
        legend=True,
    )

    fig.set_title(f"Distribution of v{col}")
    fig.set_xlabel(f"v{col}")
    fig.set_ylabel("Density")
    plt.show()

### Amount

In [ ]:
col_to_plot = "amount"

fig = sns.histplot(
    df,
    x=col_to_plot,
    stat="density",
    hue="is_fraud",
    kde=True,
    common_norm=False,
    legend=True,
)

fig.set_xlim(0, 250)
fig.set_title(f"Distribution of transactions {col_to_plot} lees than 100")
fig.set_xlabel(f"{col_to_plot.capitalize()}")
fig.set_ylabel("Density")
plt.show()

### card_id

In [ ]:
# Top 20 use card_id
n = 20
col = "card_id"
top_n = df[col].value_counts().sort_values(ascending=False).head(n)

fig, ax = plt.subplots(layout="constrained")
sns.barplot(top_n)

ax.tick_params("x", rotation=45)
ax.set_ylabel("Count")
ax.set_title(f"Top {n} most frequent card IDs")
plt.show()

In [ ]:
# Number of card id by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("card_id")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_cards")
)

### merchant_id

In [ ]:
# Top 20 merchant_id
n = 20
col = "merchant_id"
top_n = df[col].value_counts().sort_values(ascending=False).head(n)

fig, ax = plt.subplots(layout="constrained")
sns.barplot(top_n)
for container in ax.containers:
    ax.bar_label(container)
ax.tick_params("x", rotation=45)
ax.set_ylabel("Count")
ax.set_title(f"Top {n} merchants by number of transactions")
plt.show()

In [ ]:
# Number of merchant id by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("merchant_id")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_merchants")
)

### merchant_category_code

In [ ]:
fig, ax = plt.subplots(layout="constrained")

sns.countplot(
    data=df,
    x="merchant_category_code",
    stat="percent",
    hue="is_fraud",
    order=df["merchant_category_code"].value_counts().index,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

ax.set_ylabel("Percentage")
ax.set_title("Merchant category by number of transactions")
plt.show()

In [ ]:
# Number of merchant category by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("merchant_category_code")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_merchant_category_code")
)

### merchant_country

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9), layout="constrained")
sns.countplot(
    data=df,
    x="merchant_country",
    stat="percent",
    hue="is_fraud",
    order=df["merchant_country"].value_counts().index,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f%%")

ax.set_ylabel("Percentage")
ax.set_title("Merchant country by number of transactions")
plt.show()

In [ ]:
# Number of merchant country by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("merchant_country")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_merchant_country")
)

### currency

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9), layout="constrained")
sns.countplot(
    data=df,
    x="currency",
    stat="percent",
    hue="is_fraud",
    order=df["currency"].value_counts().index,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

ax.set(
    ylabel="Percentage",
    title="Currency by number of transactions",
)
plt.show()

In [ ]:
# Number of currency by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("currency")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_currency")
)

### device_type

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9), layout="constrained")
sns.countplot(
    data=df,
    x="device_type",
    stat="percent",
    hue="is_fraud",
    order=df["device_type"].value_counts().index,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

ax.set(
    ylabel="Percentage",
    title="Type of device by number of transactions",
)
plt.show()

In [ ]:
# Number of device type by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("device_type")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_device_type")
)

### entry_mode

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9), layout="constrained")
sns.countplot(
    data=df,
    x="entry_mode",
    stat="percent",
    hue="is_fraud",
    order=df["entry_mode"].value_counts().index,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

ax.set(
    ylabel="Percentage",
    title="Entry mode by number of transactions",
)
plt.show()

In [ ]:
# Number of entry mode by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("entry_mode")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_entry_mode")
)

### channel

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9), layout="constrained")
sns.countplot(
    data=df,
    x="channel",
    stat="percent",
    hue="is_fraud",
    order=df["channel"].value_counts().index,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

ax.set(
    ylabel="Percentage",
    title="Channel by number of transactions",
)
plt.show()

In [ ]:
# Number of channel by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("channel")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_channel")
)

### is_international

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9), layout="constrained")
sns.countplot(
    data=df,
    x="is_international",
    stat="percent",
    hue="is_fraud",
    order=df["is_international"].value_counts().index,
    ax=ax,
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%")

ax.set(
    ylabel="Percentage",
    title="Number of international transactions",
)
plt.show()

In [ ]:
# Number of international transactions by fraud count
(
    df.astype({"is_fraud": "int"})
    .groupby("is_international")["is_fraud"]
    .sum()
    .value_counts()
    .sort_index()
    .rename_axis("number_of_frauds")
    .reset_index(name="number_of_is_international")
)